In [ ]:
# --- parameters (patch_notebook_params.py) ---
MAX_EPOCHS = 2000
N_SAMPLES = 10           # run_metrics.py forces 50 for generative methods
RESET_TRAINING = False
CUDA_VISIBLE_DEVICES = "0"
METRICS_CSV = "results/metrics.csv"
SKIP_TRAINING = False    # run_metrics.py sets this True: load weights from
                         # the checkpoint directly instead of calling
                         # trainer.fit(), which can silently retrain for the
                         # full schedule if the checkpoint does not cleanly
                         # resume to exactly MAX_EPOCHS.


In [ ]:
# --- epoch heartbeat (patch_notebook_params.py) ---
import pytorch_lightning as _pl

class EpochHeartbeat(_pl.Callback):
    """Prints one clear progress line every `every_n_epochs` epochs, so
    sbatch logs show training progress without the noise of a per-batch
    tqdm progress bar (which doesn't render well once redirected to a
    plain log file).    """

    def __init__(self, every_n_epochs: int = 1):
        self.every_n_epochs = every_n_epochs

    def on_train_epoch_end(self, trainer, pl_module):
        epoch = trainer.current_epoch + 1
        if epoch % self.every_n_epochs != 0 and epoch != trainer.max_epochs:
            return
        parts = []
        for k, v in sorted(trainer.callback_metrics.items()):
            try:
                parts.append(f'{k}={float(v):.4f}')
            except (TypeError, ValueError):
                pass
        print(f'[progress] epoch {epoch}/{trainer.max_epochs} | ' + ' | '.join(parts), flush=True)


# Direct UNet — SSH Reconstruction from Masked Observations

Simplified version of the consistency model notebook.
No generative model — the UNet learns a **direct supervised mapping**:

$$\hat{x} = \text{UNet}(y_\text{filled},\; m,\; \nabla_x J)$$

| Input | Shape | Description |
|-------|-------|-------------|
| $y_\text{filled}$ | `(B, C, H, W)` | Observations with NaN → 0 |
| $m$ | `(B, C, H, W)` | Binary mask (1 = obs available, 0 = NaN) |

The network is trained with MSE loss against the ground truth $x$.


## 🛠️ Setup

In [ ]:
!nvidia-smi


In [ ]:
import os; os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES

### Imports

In [ ]:
import json
import os
import zipfile
import glob
from dataclasses import asdict, dataclass
from typing import Any, Callable, List, Optional, Tuple, Union

import torch
from einops import rearrange
from einops.layers.torch import Rearrange
import pytorch_lightning as pl
from pytorch_lightning import LightningDataModule, LightningModule, Trainer, seed_everything
from pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from matplotlib import pyplot as plt
from torch import Tensor, nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torchinfo import summary

import sys
sys.path.append('../../..')


## 🧠 Implementation

### DataModule

In [ ]:
import sys
import pyinterp
import pyinterp.fill
import pyinterp.backends.xarray
from src.dataloader_SSH import *

import matplotlib.pyplot as plt
import torch
import itertools

lon_min = -66.3
lon_max = -53.5
lat_min = 31.7
lat_max = 44.5

datadir = "../../../data"

TrainingItem = namedtuple('TrainingItem', ['input', 'tgt'])

def remove_nan(da):
    da["lon"] = da.lon.assign_attrs(units="degrees_east")
    da["lat"] = da.lat.assign_attrs(units="degrees_north")
    da.transpose("lon", "lat", "time")[:, :] = pyinterp.fill.gauss_seidel(
        pyinterp.backends.xarray.Grid3D(da)
    )[1]
    return da

def load_altimetry_data(path, obs_from_tgt=False):
    ds = (
        xr.open_dataset(path)
        .load()
        .assign(
            input=lambda ds: ds.nadir_obs,
            tgt=lambda ds: remove_nan(ds.ssh),
        )
    )
    if obs_from_tgt:
        ds = ds.assign(input=ds.tgt.where(np.isfinite(ds.input), np.nan))
    return (
        ds[[*TrainingItem._fields]]
        .transpose("time", "lat", "lon")
        .to_array()
    )

datamodule = BaseDataModule(
    input_da=load_altimetry_data(datadir + "/natl_gf_w_5nadirs_swot.nc"),
    domains={
        'train': {'time': slice('2013-02-24', '2013-09-30')},
        'val':   {'time': slice('2012-12-15', '2013-02-24')},
        'test':  {'time': slice('2012-10-01', '2012-12-20')},
    },
    xrds_kw={
        'patch_dims':    {'time': 15, 'lat': 128, 'lon': 128},
        'strides':       {'time': 1,  'lat': 128, 'lon': 128},
        'domain_limits': dict(lon=slice(lon_min, lon_max),
                              lat=slice(lat_min, lat_max)),
    },
    dl_kw={'batch_size': 4, 'num_workers': 1},
    grad=False,
    resize_factor=2,
)
datamodule.setup()

# Quick sanity plot
xr.Dataset(
    data_vars={'ssh': (('time', 'lat', 'lon'), datamodule.train_ds[0].tgt)},
    coords={
        'time': np.arange(15),
        'lon':  np.arange(lon_min, lon_max, 0.1),
        'lat':  np.arange(lat_min, lat_max, 0.1),
    },
).ssh.plot(col='time', col_wrap=15)


### Modules (UNet blocks — identiques au notebook de référence)

In [ ]:
def GroupNorm(channels: int) -> nn.GroupNorm:
    return nn.GroupNorm(num_groups=min(32, channels // 4), num_channels=channels)


class SelfAttention(nn.Module):
    def __init__(self, in_channels: int, out_channels: int,
                 n_heads: int = 8, dropout: float = 0.3) -> None:
        super().__init__()
        self.dropout = dropout
        self.qkv_projection = nn.Sequential(
            GroupNorm(in_channels),
            nn.Conv2d(in_channels, 3 * in_channels, kernel_size=1, bias=False),
            Rearrange("b (i h d) x y -> i b h (x y) d", i=3, h=n_heads),
        )
        self.output_projection = nn.Sequential(
            Rearrange("b h l d -> b l (h d)"),
            nn.Linear(in_channels, out_channels, bias=False),
            Rearrange("b l d -> b d l"),
            GroupNorm(out_channels),
            nn.Dropout1d(dropout),
        )
        self.residual_projection = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x: Tensor) -> Tensor:
        q, k, v = self.qkv_projection(x).unbind(dim=0)
        output = F.scaled_dot_product_attention(
            q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=False
        )
        output = self.output_projection(output)
        output = rearrange(output, "b c (x y) -> b c x y", x=x.shape[-2], y=x.shape[-1])
        return output + self.residual_projection(x)


class UNetBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int,
                 noise_level_channels: int, dropout: float = 0.3) -> None:
        super().__init__()
        self.input_projection = nn.Sequential(
            GroupNorm(in_channels), nn.SiLU(),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding="same"),
            nn.Dropout2d(dropout),
        )
        self.noise_level_projection = nn.Sequential(
            nn.SiLU(),
            nn.Conv2d(noise_level_channels, out_channels, kernel_size=1),
        )
        self.output_projection = nn.Sequential(
            GroupNorm(out_channels), nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding="same"),
            nn.Dropout2d(dropout),
        )
        self.residual_projection = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x: Tensor, cond: Tensor) -> Tensor:
        h = self.input_projection(x)
        h = h + self.noise_level_projection(cond)
        return self.output_projection(h) + self.residual_projection(x)


class UNetBlockWithSelfAttention(nn.Module):
    def __init__(self, in_channels: int, out_channels: int,
                 noise_level_channels: int, n_heads: int = 8, dropout: float = 0.3) -> None:
        super().__init__()
        self.unet_block = UNetBlock(in_channels, out_channels, noise_level_channels, dropout)
        self.self_attention = SelfAttention(out_channels, out_channels, n_heads, dropout)

    def forward(self, x: Tensor, cond: Tensor) -> Tensor:
        return self.self_attention(self.unet_block(x, cond))


class Downsample(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.projection = nn.Sequential(
            Rearrange("b c (h ph) (w pw) -> b (c ph pw) h w", ph=2, pw=2),
            nn.Conv2d(4 * channels, channels, kernel_size=1),
        )
    def forward(self, x: Tensor) -> Tensor:
        return self.projection(x)


class Upsample(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.projection = nn.Sequential(
            nn.Upsample(scale_factor=2.0, mode="nearest"),
            nn.Conv2d(channels, channels, kernel_size=3, padding="same"),
        )
    def forward(self, x: Tensor) -> Tensor:
        return self.projection(x)


### UNet direct

Le réseau prend en entrée :
- `y_filled` : observations avec NaN → 0  `(B, C, H, W)`
- `mask` : masque binaire 0/1  `(B, C, H, W)`

Les deux sont projetés séparément puis **additionnés** avant le premier bloc encodeur (même logique que `input_projection` + `grad_j_projection` dans le notebook de référence).

Il n'y a **pas d'embedding temporel** : le conditioning est remplacé par un vecteur nul de même forme, ignoré par les blocs.

In [ ]:
@dataclass
class DirectUNetConfig:
    channels: int = 15                               # C — nb de pas de temps SSH
    cond_channels: int = 1                           # dimension du vecteur de conditioning (inutilisé → 1)
    n_heads: int = 8
    top_blocks_channels: Tuple[int, ...] = (128, 128)
    top_blocks_n_blocks_per_resolution: Tuple[int, ...] = (2, 2)
    top_blocks_has_resampling: Tuple[bool, ...] = (True, True)
    top_blocks_dropout: Tuple[float, ...] = (0.0, 0.0)
    mid_blocks_channels: Tuple[int, ...] = (256, 512)
    mid_blocks_n_blocks_per_resolution: Tuple[int, ...] = (4, 4)
    mid_blocks_has_resampling: Tuple[bool, ...] = (True, False)
    mid_blocks_dropout: Tuple[float, ...] = (0.0, 0.0)


class DirectUNet(nn.Module):
    """
    UNet de reconstruction directe :

        x_hat = UNet(y_filled, mask)

    - y_filled  (B, C, H, W) : observations, NaN remplacés par 0
    - mask      (B, C, H, W) : masque binaire float (1 = obs disponible)

    Les deux signaux sont projetés séparément (C→D) puis additionnés,
    exactement comme input_projection + grad_j_projection dans le notebook
    de référence.  Le conditioning des blocs UNetBlock est un tenseur nul
    de forme (B, 1, 1, 1) — il n'influence pas le calcul mais respecte
    l'interface des blocs réutilisés tels quels.
    """
    def __init__(self, config: DirectUNetConfig) -> None:
        super().__init__()
        self.config = config
        D = config.top_blocks_channels[0]

        self.obs_projection  = nn.Conv2d(config.channels, D, kernel_size=3, padding="same")
        self.mask_projection = nn.Conv2d(config.channels, D, kernel_size=3, padding="same")

        # Conditioning nul : 1 canal → projeté vers la taille attendue par les blocs
        COND = config.cond_channels  # 1

        self.top_encoder_blocks = self._make_encoder_blocks(
            config.top_blocks_channels + config.mid_blocks_channels[:1],
            config.top_blocks_n_blocks_per_resolution,
            config.top_blocks_has_resampling,
            config.top_blocks_dropout,
            self._make_top_block,
        )
        self.mid_encoder_blocks = self._make_encoder_blocks(
            config.mid_blocks_channels + config.mid_blocks_channels[-1:],
            config.mid_blocks_n_blocks_per_resolution,
            config.mid_blocks_has_resampling,
            config.mid_blocks_dropout,
            self._make_mid_block,
        )
        self.mid_decoder_blocks = self._make_decoder_blocks(
            config.mid_blocks_channels + config.mid_blocks_channels[-1:],
            config.mid_blocks_n_blocks_per_resolution,
            config.mid_blocks_has_resampling,
            config.mid_blocks_dropout,
            self._make_mid_block,
        )
        self.top_decoder_blocks = self._make_decoder_blocks(
            config.top_blocks_channels + config.mid_blocks_channels[:1],
            config.top_blocks_n_blocks_per_resolution,
            config.top_blocks_has_resampling,
            config.top_blocks_dropout,
            self._make_top_block,
        )
        self.output_projection = nn.Conv2d(D, config.channels, kernel_size=3, padding="same")

    def forward(self, y_filled: Tensor, mask: Tensor) -> Tensor:
        """
        Parameters
        ----------
        y_filled : (B, C, H, W)  observations avec NaN→0
        mask     : (B, C, H, W)  masque binaire float

        Returns
        -------
        x_hat : (B, C, H, W)  reconstruction de x
        """
        B = y_filled.shape[0]
        h = self.obs_projection(y_filled) + self.mask_projection(mask)

        # Conditioning nul — forme (B, 1, 1, 1) compatible avec les blocs
        cond = torch.zeros(B, self.config.cond_channels, 1, 1,
                           device=y_filled.device, dtype=y_filled.dtype)

        top_skips = []
        for block in self.top_encoder_blocks:
            if isinstance(block, UNetBlock):
                h = block(h, cond)
                top_skips.append(h)
            else:
                h = block(h)

        mid_skips = []
        for block in self.mid_encoder_blocks:
            if isinstance(block, UNetBlockWithSelfAttention):
                h = block(h, cond)
                mid_skips.append(h)
            else:
                h = block(h)

        for block in self.mid_decoder_blocks:
            if isinstance(block, UNetBlockWithSelfAttention):
                h = torch.cat((h, mid_skips.pop()), dim=1)
                h = block(h, cond)
            else:
                h = block(h)

        for block in self.top_decoder_blocks:
            if isinstance(block, UNetBlock):
                h = torch.cat((h, top_skips.pop()), dim=1)
                h = block(h, cond)
            else:
                h = block(h)

        return self.output_projection(h)

    # ---- builder helpers (identiques au notebook de référence) ----
    def _make_encoder_blocks(self, channels, n_blocks, has_resampling, dropout, block_fn):
        blocks = nn.ModuleList()
        for idx, (ic, oc) in enumerate(zip(channels[:-1], channels[1:])):
            for _ in range(n_blocks[idx]):
                blocks.append(block_fn(ic, oc, dropout[idx]))
                ic = oc
            if has_resampling[idx]:
                blocks.append(Downsample(oc))
        return blocks

    def _make_decoder_blocks(self, channels, n_blocks, has_resampling, dropout, block_fn):
        blocks = nn.ModuleList()
        for idx, (oc, ic) in enumerate(list(zip(channels[:-1], channels[1:]))[::-1]):
            if has_resampling[::-1][idx]:
                blocks.append(Upsample(ic))
            inner = []
            for _ in range(n_blocks[::-1][idx]):
                inner.append(block_fn(ic * 2, oc, dropout[::-1][idx]))
                oc = ic
            blocks.extend(inner[::-1])
        return blocks

    def _make_top_block(self, ic, oc, dropout):
        return UNetBlock(ic, oc, self.config.cond_channels, dropout)

    def _make_mid_block(self, ic, oc, dropout):
        return UNetBlockWithSelfAttention(
            ic, oc, self.config.cond_channels, self.config.n_heads, dropout
        )

    def save_pretrained(self, pretrained_path: str) -> None:
        os.makedirs(pretrained_path, exist_ok=True)
        with open(os.path.join(pretrained_path, "config_direct_unet.json"), mode="w") as f:
            json.dump(asdict(self.config), f)
        torch.save(self.state_dict(), os.path.join(pretrained_path, "model_direct_unet.pt"))

    @classmethod
    def from_pretrained(cls, pretrained_path: str) -> "DirectUNet":
        with open(os.path.join(pretrained_path, "config_direct_unet.json"), mode="r") as f:
            config_dict = json.load(f)
        config = DirectUNetConfig(**config_dict)
        model = cls(config)
        state_dict = torch.load(
            os.path.join(pretrained_path, "model_direct_unet.pt"),
            map_location=torch.device("cpu"),
        )
        model.load_state_dict(state_dict)
        return model


summary(
    DirectUNet(DirectUNetConfig()),
    input_size=((1, 15, 128, 128),   # y_filled
                (1, 15, 128, 128)),  # mask
)


### LitDirectUNet

In [ ]:
@dataclass
class LitDirectUNetConfig:
    lr: float = 1e-4
    betas: Tuple[float, float] = (0.9, 0.995)
    lr_scheduler_start_factor: float = 1e-5
    lr_scheduler_iters: int = 10_000


class LitDirectUNet(LightningModule):
    """
    Lightning wrapper pour la reconstruction directe par UNet.

    training_step :
        1. Construit y_filled et mask à partir de batch.input (NaN → 0)
        2. Prédit x_hat = model(y_filled, mask)
        3. Loss = MSE(x_hat, batch.tgt)  sur tous les pixels
           (on supervise sur toute la grille, y compris là où il n'y a pas d'obs)
    """

    def __init__(
        self,
        model: nn.Module,
        config: LitDirectUNetConfig = LitDirectUNetConfig(),
    ) -> None:
        super().__init__()
        self.model = model
        self.config = config

    def _prepare_inputs(self, y: Tensor) -> Tuple[Tensor, Tensor]:
        """Retourne (y_filled, mask) depuis y qui contient des NaN."""
        mask     = y.isfinite().to(y.dtype)        # (B, C, H, W) float 0/1
        y_filled = y.nan_to_num(0.0)               # (B, C, H, W) NaN→0
        return y_filled, mask

    def training_step(self, batch, batch_idx: int):
        if isinstance(batch, list):
            batch = batch[0]

        y_filled, mask = self._prepare_inputs(batch.input)
        x_hat = self.model(y_filled, mask)
        loss  = F.mse_loss(x_hat, batch.tgt)

        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx: int):
        if isinstance(batch, list):
            batch = batch[0]

        y_filled, mask = self._prepare_inputs(batch.input)
        x_hat = self.model(y_filled, mask)
        loss  = F.mse_loss(x_hat, batch.tgt)
        rmse  = loss.sqrt()

        self.log_dict({"val_loss": loss, "val_rmse": rmse}, prog_bar=True)
        return loss

    def configure_optimizers(self):
        opt = torch.optim.Adam(
            self.model.parameters(),
            lr=self.config.lr,
            betas=self.config.betas,
        )
        sched = torch.optim.lr_scheduler.LinearLR(
            opt,
            start_factor=self.config.lr_scheduler_start_factor,
            end_factor=1.0,
            total_iters=self.config.lr_scheduler_iters,
        )
        return [opt], [{"scheduler": sched, "interval": "step"}]


## 🚀 Training

In [ ]:
def is_valid_checkpoint(path: str) -> bool:
    try:
        with zipfile.ZipFile(path, 'r') as zf:
            zf.testzip()
        return True
    except Exception:
        return False

def find_best_valid_checkpoint(ckpt_dir: str) -> Optional[str]:
    if not os.path.isdir(ckpt_dir):
        return None
    last_ckpt = os.path.join(ckpt_dir, "last.ckpt")
    if os.path.exists(last_ckpt) and is_valid_checkpoint(last_ckpt):
        print(f"✅ Checkpoint valide : {last_ckpt}")
        return last_ckpt
    all_ckpts = sorted(
        [p for p in glob.glob(os.path.join(ckpt_dir, "*.ckpt"))
         if "last" not in os.path.basename(p)],
        key=lambda p: float(p.split("val_loss=")[-1].replace(".ckpt", ""))
        if "val_loss=" in p else float("inf"),
    )
    for ckpt_path in all_ckpts:
        if is_valid_checkpoint(ckpt_path):
            print(f"✅ Checkpoint valide : {ckpt_path}")
            return ckpt_path
    print("❌ Aucun checkpoint valide. Démarrage depuis zéro.")
    return None


@dataclass
class TrainingConfig:
    unet_config: DirectUNetConfig
    lit_config: LitDirectUNetConfig
    trainer: Trainer
    seed: int = 42
    log_dir: str = "logs_direct_unet"
    log_version: str = "v0"
    resume_ckpt_path: Optional[str] = None
    reload: bool = False

    @property
    def run_dir(self) -> str:
        return os.path.join(self.log_dir, self.log_version)

    @property
    def model_ckpt_path(self) -> str:
        return os.path.join(self.run_dir, "best_model")

    @property
    def lightning_ckpt_dir(self) -> str:
        return os.path.join(self.run_dir, "checkpoints")


def run_training(config: TrainingConfig) -> None:
    seed_everything(config.seed)

    if not config.reload:
        model = DirectUNet(config.unet_config)
    else:
        model = DirectUNet.from_pretrained(config.model_ckpt_path)

    lit = LitDirectUNet(model, config.lit_config)

    config.trainer.callbacks.append(EpochHeartbeat(every_n_epochs=1))
    if SKIP_TRAINING:
        if config.resume_ckpt_path is None:
            raise RuntimeError(
                f"SKIP_TRAINING=True but no checkpoint found in {config.lightning_ckpt_dir} -- "
                "run training first (submit_train.sbatch) before computing metrics."
            )
        print(f'[TRAINING] SKIP_TRAINING=True -- loading weights from {config.resume_ckpt_path} directly (trainer.fit() not called)', flush=True)
        _ckpt_state = torch.load(config.resume_ckpt_path, map_location='cpu')
        lit.load_state_dict(_ckpt_state['state_dict'])
    else:
        print(f'[TRAINING] resume_ckpt={config.resume_ckpt_path!r} | MAX_EPOCHS={MAX_EPOCHS}', flush=True)
        config.trainer.fit(lit, datamodule, ckpt_path=config.resume_ckpt_path)

    lit.model.save_pretrained(config.model_ckpt_path)
    print(f"✅ Model saved to: {config.model_ckpt_path}")


### Run Training

In [ ]:
LOG_DIR     = "logs_direct_unet"
CKPT_DIR    = os.path.join(LOG_DIR, "checkpoints")
resume_ckpt = find_best_valid_checkpoint(CKPT_DIR)

training_config = TrainingConfig(
    unet_config=DirectUNetConfig(),
    lit_config=LitDirectUNetConfig(lr_scheduler_iters=1000),
    log_dir=LOG_DIR,
    log_version="",
    trainer=Trainer(enable_progress_bar=False, 
        accelerator="gpu",
        max_epochs=MAX_EPOCHS,
        accumulate_grad_batches=4,
        precision="16-mixed",
        log_every_n_steps=1,
        logger=TensorBoardLogger(".", name=LOG_DIR, version=""),
        callbacks=[
            LearningRateMonitor(logging_interval="step"),
            ModelCheckpoint(
                dirpath=CKPT_DIR,
                monitor="val_loss",
                save_top_k=3,
                save_last=True,
                filename="{epoch:03d}-{step}-{val_loss:.4f}",
            ),
        ],
    ),
    resume_ckpt_path=resume_ckpt,  # was commented out -- training always restarted from scratch
)
run_training(training_config)


## 🔍 Évaluation & Visualisation

### Chargement du checkpoint

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
dtype  = torch.bfloat16 if torch.cuda.is_available() else torch.float32

MODEL_PATH = os.path.join("logs_direct_unet", "best_model")

model = DirectUNet.from_pretrained(MODEL_PATH).eval().to(device=device, dtype=dtype)


### Inférence sur un batch test

In [ ]:
def plot_ssh(images: Tensor, vmin: float = -2., vmax: float = 2., cols: int = 15) -> None:
    """Affiche un champ SSH (C, H, W) comme grille de C cartes."""
    xr.Dataset(
        data_vars={'ssh': (('time', 'lat', 'lon'), images.float().cpu())},
        coords={
            'time': np.arange(images.shape[0]),
            'lon':  np.arange(images.shape[2]),
            'lat':  np.arange(images.shape[1]),
        },
    ).ssh.plot(col='time', col_wrap=cols, vmin=vmin, vmax=vmax)


# ---- Batch de test ----
seed_everything(42)
batch = next(iter(datamodule.test_dataloader()))

y      = batch.input.to(device=device, dtype=dtype)   # (B, C, H, W)  avec NaN
x_true = batch.tgt  .to(device=device, dtype=dtype)   # (B, C, H, W)

mask     = y.isfinite().to(dtype)
y_filled = y.nan_to_num(0.0)

with torch.no_grad():
    x_hat = model(y_filled, mask)

# ---- Métriques ----
rmse = (x_hat - x_true).pow(2).mean().sqrt().item()
print(f"RMSE (test batch) : {rmse:.4f}")

# ---- Visualisation — premier échantillon du batch ----
b = 0
print("Observations y (NaN → 0) :")
plot_ssh(y_filled[b].cpu())

print("Reconstruction UNet x_hat :")
plot_ssh(x_hat[b].cpu())

print("Vérité terrain x :")
plot_ssh(x_true[b].cpu())

print("Résidu |x_hat - x| :")
plot_ssh((x_hat[b] - x_true[b]).abs().cpu(), vmin=0, vmax=0.5)


In [ ]:
import sys
sys.path.append('../..')   # -> consistency/
from spectral_utils import radial_psd_2d, psd_spectral_score, resolved_scale
import numpy as np
import pandas as pd

DX_KM = 0.1 * 111   # 0.1 deg grid spacing -> km

# ── Full test-set evaluation ─────────────────────────────────────────────
# Loops over EVERY batch of datamodule.test_dataloader() (not just the first
# one used for the illustrative visualisation above), computing RMSE and the
# spectral resolved scale (lambda_x, via the SAME spectral_utils functions
# used by every other notebook in the suite -- previously this cell used its
# own ad-hoc PSD/lambda_x function, which computed a methodologically
# DIFFERENT quantity (raw PSD 50%-of-peak threshold on GT and reconstruction
# separately) than the rest of the suite (PSD-of-ERROR spectral score) --
# mixing the two within the same comparison table column would not be a
# valid apples-to-apples comparison). Deterministic method -- one forward
# pass per batch, no ensemble/CRPS.
_all = {'rmse': [], 'lambda_x': []}
_n_samples = 0

for _tb in datamodule.test_dataloader():
    _y_b      = _tb.input.to(device=device, dtype=dtype)
    _x_true_b = _tb.tgt.to(device=device, dtype=dtype)
    _mask_b     = _y_b.isfinite().to(dtype)
    _y_filled_b = _y_b.nan_to_num(0.0)
    with torch.no_grad():
        _x_hat_b = model(_y_filled_b, _mask_b)

    for _bi in range(_x_true_b.shape[0]):
        _gt       = _x_true_b[_bi].float().cpu().numpy()
        _x_hat_np = _x_hat_b[_bi].float().cpu().numpy()

        _gt_std = np.nanstd(_gt)
        if _gt_std <= 0:
            continue
        _rmse_val = float(np.sqrt(np.nanmean((_x_hat_np - _gt) ** 2)))

        _gt_f   = np.nan_to_num(_gt,       nan=0.0)
        _pred_f = np.nan_to_num(_x_hat_np, nan=0.0)
        _lam_vals = []
        for _t in range(_gt_f.shape[0]):
            _wl, _, _, _spec = psd_spectral_score(_pred_f[_t], _gt_f[_t], dx=DX_KM)
            _lam = resolved_scale(_wl, _spec, threshold=0.5)
            if not np.isnan(_lam):
                _lam_vals.append(_lam)

        _all['rmse'].append(_rmse_val)
        if _lam_vals:
            _all['lambda_x'].append(float(np.mean(_lam_vals)))
        _n_samples += 1

print(f"Evaluated {_n_samples} test samples (full assimilation window each) across the full test set")

def _agg(vals):
    a = np.asarray(vals, dtype=float)
    a = a[~np.isnan(a)]
    return (float(np.mean(a)), float(np.std(a))) if len(a) else (np.nan, np.nan)

_rmse_m, _rmse_s = _agg(_all['rmse'])
_lam_m, _lam_s   = _agg(_all['lambda_x'])

row_du = {
    'Method'  : 'DirectUNet (full test set)',
    'RMSE'    : f'{_rmse_m:.4f} $\\pm$ {_rmse_s:.4f}',
    'lambda_x': f'{_lam_m:.2f} $\\pm$ {_lam_s:.2f}' if not np.isnan(_lam_m) else '?',
}

df_metrics = pd.DataFrame([row_du]).set_index('Method')
print(f"## Performance Assessment -- DirectUNet (full test set, n={_n_samples} samples)")
print()
display(df_metrics)

df_metrics

In [ ]:
# --- metrics serialization (patch_notebook_params.py) ---
import os
os.makedirs(os.path.dirname(METRICS_CSV) or '.', exist_ok=True)
_df_out = df_metrics.reset_index() if df_metrics.index.name == 'Method' else df_metrics
_df_out.to_csv(METRICS_CSV, index=False)
print(f'Metrics written to {METRICS_CSV}')
